# Visualisation conceptuelle de `SpatialDifferenceFusion`

Sélection d’un amas dans les données prétraitées et affichage des **quatre composantes** de la concaténation spatiale, directement dans l’espace des images :

\[
\operatorname{combined}=\operatorname{concat}\left(RM,\ XM,\ |R-X|M,\ (R\odot X)M\right).
\]

**Important :** il s’agit d’une illustration sur les intensités et **non** des cartes de features apprises par les encodeurs. La soustraction et le produit dépendent fortement de la normalisation de `R` et `X` ; ils ne représentent pas directement une différence physique entre flux radio et X-ray.

Prérequis : `numpy`, `matplotlib`, `ipywidgets`. `astropy` est nécessaire seulement pour lire les FITS. Le notebook ne recalcule pas la reprojection WCS : les tableaux prétraités doivent déjà être alignés et de même dimension.

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# À ADAPTER : le dossier produit par ton preprocessing
PROCESSED_DIR = Path('../data/processed').expanduser().resolve()

# Si un fichier prétraité contient plusieurs tableaux, voici les noms reconnus.
RADIO_KEYS = ('raw', 'radio', 'radio_image', 'image_radio')
XRAY_KEYS = ('chandra', 'xray', 'x_ray', 'xray_image')
MASK_KEYS = ('chandra_mask', 'mask_chandra', 'xray_mask', 'mask', 'coverage_mask')

# Répertoires communs aux trois types de cartes ; ne pas inclure raw/chandra dans l'identifiant d'amas.
CLUSTER_PREFIX = 'PSZ2'  # Seuls ces amas sont proposés dans le sélecteur.

ROLE_TOKENS = {'raw','radio','chandra','xray','x_ray','mask','coverage','processed','preprocessed','image','images'}
print('Processed directory:', PROCESSED_DIR)
print('Exists:', PROCESSED_DIR.exists())

Processed directory: /Users/antoinegdrn/PH_Project_I/clusterprep/data/processed
Exists: True


## 1. Découverte des amas PSZ2

Le sélecteur recense uniquement les fichiers ou dossiers dont le nom commence par **PSZ2**. Il extrait un identifiant commun (par exemple `PSZ2G179.09+60.12`) depuis les noms `PSZ2G179.09+60.12_raw.npy`, `PSZ2G179.09+60.12_chandra.npy` et `PSZ2G179.09+60.12_mask.npy`. Les trois modalités sont alors regroupées sous le même amas.

Les bundles `.npz` sont également pris en charge lorsqu'ils contiennent des tableaux nommés `raw`, `chandra`, `mask` (ou leurs alias). Le chargement exige des tableaux alignés et un masque explicite ou inférable des NaN.


In [2]:
SUPPORTED = {'.npz', '.npy', '.fits', '.fit', '.fts'}

def _read_array(path):
    path = Path(path)
    if path.suffix.lower() == '.npy':
        return np.load(path, allow_pickle=False)
    if path.suffix.lower() == '.npz':
        with np.load(path, allow_pickle=False) as bundle:
            return {key: bundle[key] for key in bundle.files}
    if path.suffix.lower() in {'.fits', '.fit', '.fts'}:
        from astropy.io import fits
        with fits.open(path, memmap=False) as hdul:
            return np.asarray(hdul[0].data)
    raise ValueError(f'Extension non supportée : {path.suffix}')

def _as_image(arr, name):
    arr = np.asarray(arr, dtype=np.float32).squeeze()
    if arr.ndim != 2:
        raise ValueError(f'{name}: tableau 2D attendu, reçu {arr.shape}')
    return arr

def _get_named(mapping, names):
    by_lower = {str(k).lower(): k for k in mapping}
    for name in names:
        if name.lower() in by_lower:
            return mapping[by_lower[name.lower()]]
    return None

def _role_for_stem(stem):
    s = stem.lower()
    # Les noms de masques doivent être identifiés avant « chandra ».
    if 'mask' in s or 'coverage' in s:
        return 'mask'
    if re.search(r'(^|[_\-.])(raw|radio)(?=$|[_\-.])', s):
        return 'radio'
    if re.search(r'(^|[_\-.])(chandra|xray|x_ray)(?=$|[_\-.])', s):
        return 'xray'
    return None

def _cluster_id(file, root):
    """Récupère l'identifiant PSZ2..., même avec des suffixes _raw/_chandra/_mask.

    Exemples :
        PSZ2G179.09+60.12_raw.npy -> PSZ2G179.09+60.12
        PSZ2G179.09+60.12_chandra_mask.npy -> PSZ2G179.09+60.12
    """
    # La recherche se fait dans le nom, puis dans ses parents, si besoin.
    for name in [file.stem] + [p.name for p in file.parents if p != root.parent]:
        match = re.search(
            r'(PSZ2[A-Za-z]?\d{1,3}\.\d{1,3}[+-]\d{1,3}\.\d{1,3})',
            name,
            flags=re.IGNORECASE,
        )
        if match:
            return match.group(1).upper()
        # Fallback si identifiant atypique : coupe avant un suffixe de modalité.
        match = re.match(
            r'(PSZ2[^_\s]+?)(?=_(?:raw|radio|chandra|xray|mask|coverage|processed|preprocessed)(?:_|$))',
            name,
            flags=re.IGNORECASE,
        )
        if match:
            return match.group(1).upper()
        if name.upper().startswith('PSZ2') and name == file.stem:
            return name.upper()
    return None

def _extract_pack(file):
    obj = _read_array(file)
    if isinstance(obj, dict):
        radio = _get_named(obj, RADIO_KEYS)
        xray = _get_named(obj, XRAY_KEYS)
        mask = _get_named(obj, MASK_KEYS)
        if radio is None or xray is None:
            return None
        return radio, xray, mask
    arr = np.asarray(obj)
    if arr.ndim == 3 and arr.shape[0] == 3:
        return arr[0], arr[1], arr[2]
    if arr.ndim == 3 and arr.shape[-1] == 3:
        return arr[..., 0], arr[..., 1], arr[..., 2]
    return None

def discover_clusters(root):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f'Dossier introuvable : {root}')
    files = sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED and (p.stem.upper().startswith(CLUSTER_PREFIX) or any(parent.name.upper().startswith(CLUSTER_PREFIX) for parent in p.parents if parent != root.parent)))
    packs, grouped = {}, {}
    for path in files:
        role = _role_for_stem(path.stem)
        ident = _cluster_id(path, root)
        if not ident:
            continue
        if path.suffix.lower() in {'.npy','.npz'}:
            # Les packs sont repérés au chargement, pas nécessairement au scan.
            if role is None:
                packs.setdefault(ident, []).append(path)
        if role:
            grouped.setdefault(ident, {}).setdefault(role, []).append(path)
    catalog = {}
    for ident, rolemap in grouped.items():
        if 'radio' in rolemap:
            catalog[ident] = {'type':'separate', 'roles':rolemap}
    for ident, paths in packs.items():
        if ident not in catalog:
            catalog[ident] = {'type':'pack_candidates', 'paths':paths}
    return dict(sorted(catalog.items())), len(files)

catalog, file_count = discover_clusters(PROCESSED_DIR)
print(f'{file_count} fichiers examinés ; {len(catalog)} amas/candidats trouvés')
print('Exemples :', list(catalog)[:12])
if not catalog:
    print('Exemples de fichiers présents :', [p.name for p in sorted(PROCESSED_DIR.rglob('*')) if p.is_file()][:15])

377 fichiers examinés ; 207 amas/candidats trouvés
Exemples : ['PSZ2G023.17+86.71', 'PSZ2G031.93+78.71', 'PSZ2G040.58+77.12', 'PSZ2G045.13+67.78', 'PSZ2G045.87+57.70', 'PSZ2G046.88+56.48', 'PSZ2G048.10+57.16', 'PSZ2G048.75+53.18', 'PSZ2G049.18+65.05', 'PSZ2G049.32+44.37', 'PSZ2G050.46+67.54', 'PSZ2G052.08+46.13']


## 2. Chargement, contrôle de l'alignement et normalisation

Par défaut, `normalize = percentiles` met **séparément** radio et X-ray dans l’intervalle `[0,1]` sur les pixels de couverture valides. Cette normalisation est commode pour visualiser les composantes, mais ne conserve pas une comparaison absolue des intensités physiques. Option `none` pour travailler sur les valeurs prétraitées telles quelles, ou `shared` pour partager les bornes d’échelle des deux images.

Le masque non couvert est utilisé pour annuler les cartes X-ray et les interactions ; la composante radio est aussi multipliée par le masque uniquement afin de représenter la formule `combined` demandée. La radio elle-même reste visible hors couverture dans le panneau de contrôle.

In [13]:
def load_cluster(ident, catalog=catalog):
    entry = catalog[ident]
    paths_used = {}
    if entry['type'] == 'separate':
        rolemap = entry['roles']
        if len(rolemap['radio']) != 1 or len(rolemap.get('xray', [])) != 1:
            raise ValueError(f'Fichiers radio/CHANDRA ambigus ou manquants : {rolemap}')
        radio_path, xray_path = rolemap['radio'][0], rolemap['xray'][0]
        radio, xray = _read_array(radio_path), _read_array(xray_path)
        if isinstance(radio, dict) or isinstance(xray, dict):
            raise ValueError('Fichiers séparés doivent contenir un tableau 2D')
        paths_used = {'radio':radio_path, 'xray':xray_path}
        mask_paths = rolemap.get('mask', [])
        if len(mask_paths) > 1:
            raise ValueError(f'Plusieurs masques possibles : {mask_paths}')
        mask = _read_array(mask_paths[0]) if mask_paths else None
        if mask_paths: paths_used['mask'] = mask_paths[0]
    else:
        selected = []
        for path in entry['paths']:
            pack = _extract_pack(path)
            if pack is not None:
                selected.append((path, pack))
        if len(selected) != 1:
            raise ValueError(f'{ident}: {len(selected)} fichiers multicanaux reconnus ; attendu 1. Chemins : {entry["paths"]}')
        path, (radio, xray, mask) = selected[0]
        paths_used = {'pack':path}
    radio, xray = _as_image(radio, 'Radio'), _as_image(xray, 'CHANDRA')
    if radio.shape != xray.shape:
        raise ValueError(f'Images non alignées en taille : radio={radio.shape}, CHANDRA={xray.shape}. Reprojection nécessaire en amont.')
    if mask is None:
        # Acceptable fallback only if nonobserved pixels are explicitly NaN.
        if np.isfinite(xray).all():
            raise ValueError('Masque absent et CHANDRA entièrement fini : couverture inconnue ; masque explicite requis.')
        mask = np.isfinite(xray).astype(np.float32)
        paths_used['mask'] = 'derived from finite CHANDRA pixels'
    else:
        mask = _as_image(mask, 'Masque')
    if mask.shape != radio.shape:
        raise ValueError(f'Masque {mask.shape} incompatible avec radio {radio.shape}')
    mask = np.clip(np.nan_to_num(mask, nan=0.0), 0.0, 1.0)
    if not np.isfinite(radio).any() or not np.isfinite(xray[mask>0]).any():
        raise ValueError('Aucune donnée radio / CHANDRA valide dans cet amas')
    return radio, xray, mask, paths_used

def _normalize_image(image, valid, mode='percentiles', reference=None, p_low=1, p_high=99):
    img = np.asarray(image, dtype=np.float32)
    vals = img[valid & np.isfinite(img)]
    if vals.size == 0: return np.zeros_like(img)
    if mode == 'none':
        return np.nan_to_num(img, nan=0, posinf=0, neginf=0)
    if reference is None:
        lo, hi = np.percentile(vals, [p_low, p_high])
    else:
        lo, hi = reference
    if hi <= lo: return np.zeros_like(img)
    out = np.clip((img-lo)/(hi-lo), 0, 1)
    return np.nan_to_num(out, nan=0, posinf=0, neginf=0)

def compute_maps(radio, xray, mask, normalization='percentiles'):
    valid = mask > 0
    if normalization == 'shared':
        values = np.concatenate([radio[np.isfinite(radio)], xray[valid & np.isfinite(xray)]])
        ref = np.percentile(values, [1,99])
    else: ref = None
    r = _normalize_image(radio, np.isfinite(radio), normalization, ref)
    xr = _normalize_image(xray, valid, normalization, ref)
    m = mask.astype(np.float32)
    return r, xr, [r*m, xr*m, np.abs(r-xr)*m, r*xr*m]

def plot_cluster(ident, normalization='percentiles', show_inputs=True):
    radio, xray, mask, paths = load_cluster(ident)
    r, xr, maps = compute_maps(radio, xray, mask, normalization)
    titles = [r'$R\,M$', r'$X\,M$', r'$|R-X|\,M$', r'$(R\odot X)\,M$']
    fig, axes = plt.subplots(2 if show_inputs else 1, 4, figsize=(17, 8 if show_inputs else 4.5), constrained_layout=True)
    if not show_inputs: axes = np.asarray([axes])
    if show_inputs:
        for ax, data, name, cmap in zip(axes[0], [r, xr, mask, (mask>0).astype(float)],
                                         ['Radio (R)', 'CHANDRA (X)', 'Mask (M)', 'Coverage (M>0)'],
                                         ['inferno','magma','gray','gray']):
            im=ax.imshow(data, origin='lower', cmap=cmap, interpolation='nearest')
            ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
            fig.colorbar(im, ax=ax, fraction=.046, pad=.03)
    for ax, data, title, cmap in zip(axes[-1], maps, titles, ['inferno','magma','viridis','viridis']):
        im=ax.imshow(data, origin='lower', cmap=cmap, interpolation='nearest')
        ax.set_title(title, fontsize=15); ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=.046, pad=.03)
    fig.suptitle(f'{ident} — conceptual spatial fusion — normalization: {normalization}', fontsize=15)
    plt.show()
    print('Sources:')
    for key, value in paths.items(): print(f'  {key}: {value}')
    print(f'Coverage: {mask.mean():.1%} | shape: {radio.shape}')
    return maps

## 3. Sélection interactive

Choisis un amas et le type de normalisation. Les figures se mettent à jour automatiquement au clic sur **Afficher**. Pour exporter l'illustration, utilise le bouton de sauvegarde Matplotlib ou ajoute `fig.savefig(...)` à `plot_cluster`.

In [14]:
if not catalog:
    print('Aucun amas trouvé. Corriger PROCESSED_DIR puis réexécuter depuis le début.')
else:
    cluster_dropdown = widgets.Dropdown(options=list(catalog), description='Amas :', layout=widgets.Layout(width='75%'))
    norm_dropdown = widgets.Dropdown(options=[('Percentiles [0,1] (individuels)', 'percentiles'),
                                             ('Bornes communes [0,1]', 'shared'),
                                             ('Sans normalisation', 'none')],
                                     value='percentiles', description='Échelle :', layout=widgets.Layout(width='60%'))
    show_inputs_checkbox = widgets.Checkbox(value=True, description='Afficher aussi les 3 entrées')
    run_button = widgets.Button(description='Afficher', button_style='primary', icon='image')
    output = widgets.Output()
    def _on_click(_):
        with output:
            clear_output(wait=True)
            try:
                plot_cluster(cluster_dropdown.value, norm_dropdown.value, show_inputs_checkbox.value)
            except Exception as exc:
                print(f'Impossible de charger cet amas : {type(exc).__name__}: {exc}')
    run_button.on_click(_on_click)
    display(widgets.VBox([cluster_dropdown, norm_dropdown, show_inputs_checkbox, run_button, output]))
    _on_click(None)

## 4. Remarques scientifiques

- L’exemple utilise des **intensités normalisées**, pas les `feature maps` CNN : dans ton modèle réel les cartes ont `fusion_hidden_dim` canaux par modalité et sont produites par les projections convolutives.
- Le produit `R*X` met en évidence les régions où les deux modalités présentent simultanément de fortes valeurs ; la différence `|R-X|` signale les divergences, sans distinguer *radio fort / X-ray faible* de *radio faible / X-ray fort*.
- Pour cibler le cas des relics, tu peux ajouter expérimentalement un **contraste orienté** `max(R-X, 0)*M` dans une cinquième figure — ce n'est pas un terme de ton modèle actuel.
- La comparaison requiert une **mise en correspondance WCS et géométrique** déjà effectuée durant le preprocessing. Des tableaux de même forme ne suffisent pas à démontrer le bon alignement astrophysique.
- Un masque de couverture fractionnaire peut représenter le pourcentage de pixels observés dans chaque cellule ; ici il est appliqué directement comme dans l'expression étudiée.
